In [0]:
%run ./config

In [0]:
%run ./silver_utils

In [0]:
%run ../connection

In [0]:
# =============================================================================
# NOTEBOOK: 02a_silver_market_snapshot.py
# LAYER:    Silver
# PURPOSE:  Transform the raw coins/markets Bronze data into a clean, typed,
#           deduplicated Silver Delta table: silver/market_snapshot
#
# SOURCE:   bronze/coins_markets_raw   (Delta table, path-based)
# OUTPUT:   silver/market_snapshot     (Delta table, path-based, MERGE upsert)
#
# WHAT THIS NOTEBOOK DOES (in order):
#   1. Read the latest pipeline run's rows from Bronze coins_markets_raw
#   2. Explode the payload column (JSON array of 50 coin dicts) into one row per coin
#   3. Extract and cast every field to its correct Silver type
#   4. Apply data quality rules (drop nulls in critical fields, validate ranges)
#   5. Deduplicate within this batch (coin_id + snapshot_date)
#   6. Add Silver audit column: silver_processed_timestamp
#   7. Reorder to final Silver column list
#   8. MERGE into silver/market_snapshot (upsert — prevents duplicate rows on re-run)
#   9. Run OPTIMIZE + Z-ORDER on the Silver table
#  10. Write run log to ADLS logging/silver/
#
# SILVER TABLE: silver_market_snapshot
#   Grain: one row per coin per pipeline run (daily snapshot)
#   Primary key: coin_id + snapshot_date
 
 
# =============================================================================
# CELL 1 — SETUP: config, utils, logging
# =============================================================================
 
# %run ./silver_config      ← uncomment in Databricks; keep commented for linting
# %run ./silver_utils       ← uncomment in Databricks; keep commented for linting
 
# The above %run magic commands make all classes from silver_config.py and
# silver_utils.py available in this notebook's scope. In Databricks, %run
# executes the target file in the current notebook's Python context.
 
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, LongType, IntegerType, StringType,
    TimestampType, DateType
)
 
# Initialise config (reads ADLS account name from Key Vault)
# adls_name = dbutils.secrets.get(scope=SecretConfig.SCOPE, key=SecretConfig.KEY_ADLS_NAME)
adls_name="adlsnewhp"
init_silver_config(adls_name)
 
logger = get_logger("silver_market_snapshot")
 
logger.info("=" * 70)
logger.info("Silver: market_snapshot — START")
logger.info(f"  Run ID    : {SilverConfig.RUN_ID}")
logger.info(f"  Run TS    : {SilverConfig.RUN_TS.isoformat()}")
logger.info(f"  Bronze in : {BronzePaths.COINS_MARKETS}")
logger.info(f"  Silver out: {SilverPaths.MARKET_SNAPSHOT}")
logger.info("=" * 70)
 

In [0]:
# =============================================================================
# CELL 2 — READ BRONZE (latest batch only)
# =============================================================================
 
bronze_df, max_bronze_ts = read_bronze_incremental(
    spark, BronzePaths.COINS_MARKETS, WatermarkPaths.COINS_MARKETS,
    WatermarkPaths.WATERMARK_TABLE, logger
)
 
# Verify the columns we depend on exist before trying to use them.
# The Bronze envelope always has these fields (added by notebook 01):
assert_required_columns(
    bronze_df,
    required_cols=["payload", "pipeline_run_id", "ingestion_timestamp"],
    table_name="coins_markets_raw",
    logger=logger
)
 
logger.info(f"  Bronze schema: {bronze_df.schema.simpleString()}")
 

In [0]:
bronze_df.display()

In [0]:
# =============================================================================
# CELL 3 — EXPLODE PAYLOAD: one row per coin
#
# WHY EXPLODE?
#   The Bronze row stores the API response for all 50 coins in a single "payload"
#   column (a JSON array). In Silver, we need one row per coin so that analysts
#   can filter, aggregate, and join by coin_id.
#
# HOW EXPLODE WORKS:
#   Bronze row:  { pipeline_run_id: "abc", payload: [{id:"bitcoin",...}, {id:"ethereum",...}] }
#   After explode:
#     Row 1: { pipeline_run_id: "abc", coin: {id:"bitcoin", current_price:65000, ...} }
#     Row 2: { pipeline_run_id: "abc", coin: {id:"ethereum", current_price:3500, ...} }
#
# The "payload" column is an ARRAY of STRUCT (Autoloader inferred this schema).
# F.explode() turns that array into one row per element.
# =============================================================================
 
logger.info("CELL 3: Exploding payload array → one row per coin")
 
coins_df = (
    bronze_df
    .select(
        F.col("pipeline_run_id"),
        F.col("ingestion_timestamp"),
        F.explode(F.col("payload")).alias("coin")   # One row per coin dict
    )
)
 
logger.info(f"  Rows after explode (should be ~50 per run): {coins_df.count():,}")
 


In [0]:
coins_df.display()

In [0]:
# =============================================================================
# CELL 4 — EXTRACT AND CAST FIELDS
#
# WHY EXPLICIT CASTING?
#   Autoloader inferred types from JSON, but JSON has no type system:
#     - Numbers that look like integers might be inferred as LONG or DOUBLE
#     - Timestamps arrive as ISO 8601 strings — must be cast to TimestampType
#     - Dates must be extracted from timestamps
#   We explicitly cast every field to the Silver spec type, regardless of what
#   Autoloader inferred. This prevents "type drift" over time as the API evolves.
#
# NULL HANDLING STRATEGY:
#   - Critical fields (current_price_usd, market_cap_usd): drop row if null or <= 0
#   - Non-critical fields (total_supply, circulating_supply): allow null with a note
#   - Negative fields (price_change_24h): allow negative — it's a valid business value
# =============================================================================
 
logger.info("CELL 4: Extracting and casting all fields to Silver types")
 
# coin.field_name accesses nested struct fields using dot notation in PySpark
typed_df = (
    coins_df
    .select(
        # ── Identity ─────────────────────────────────────────────────────────
        F.col("coin.id")
         .cast(StringType())
         .alias("coin_id"),                             # Primary key part 1
 
        F.upper(F.col("coin.symbol"))
         .cast(StringType())
         .alias("symbol"),                              # Uppercase ticker (BTC, ETH)
 
        F.col("coin.name")
         .cast(StringType())
         .alias("name"),                                # Full name (Bitcoin, Ethereum)
 
        # ── Timestamps & Dates ────────────────────────────────────────────────
        # last_updated is an ISO 8601 string: "2024-11-15T06:00:00.000Z"
        # to_timestamp() parses it into a Spark TimestampType
        F.to_timestamp(F.col("coin.last_updated"))
         .alias("snapshot_timestamp"),
 
        # Extract just the date portion for partitioning and MERGE key
        F.to_date(F.col("coin.last_updated"))
         .alias("snapshot_date"),                       # Primary key part 2
 
        # ── Price & Market Cap ────────────────────────────────────────────────
        F.col("coin.current_price")
         .cast(DoubleType())
         .alias("current_price_usd"),
 
        F.col("coin.market_cap")
         .cast(LongType())
         .alias("market_cap_usd"),
 
        F.col("coin.market_cap_rank")
         .cast(IntegerType())
         .alias("market_cap_rank"),
 
        # ── Volume ───────────────────────────────────────────────────────────
        # Replace null total_volume with 0 (some newer/smaller coins have no volume)
        F.coalesce(F.col("coin.total_volume").cast(LongType()), F.lit(0))
         .alias("total_volume_24h_usd"),
 
        # ── 24h Price Range ───────────────────────────────────────────────────
        F.col("coin.high_24h")
         .cast(DoubleType())
         .alias("high_24h_usd"),
 
        F.col("coin.low_24h")
         .cast(DoubleType())
         .alias("low_24h_usd"),
 
        # ── Price Change ──────────────────────────────────────────────────────
        # Negative values are valid (coin lost value)
        F.col("coin.price_change_24h")
         .cast(DoubleType())
         .alias("price_change_24h_usd"),
 
        F.col("coin.price_change_percentage_24h")
         .cast(DoubleType())
         .alias("price_change_pct_24h"),
 
        # ── Supply ────────────────────────────────────────────────────────────
        # total_supply is null for coins with no max supply (e.g., Ethereum after EIP-1559)
        # Allow null — do not drop. Gold layer handles null total_supply gracefully.
        F.col("coin.circulating_supply")
         .cast(DoubleType())
         .alias("circulating_supply"),
 
        F.col("coin.total_supply")
         .cast(DoubleType())
         .alias("total_supply"),                        # null = unlimited supply
 
        # ── All-Time High ─────────────────────────────────────────────────────
        F.col("coin.ath")
         .cast(DoubleType())
         .alias("ath_usd"),
 
        F.to_date(F.col("coin.ath_date"))
         .alias("ath_date"),
 
        # ── Lineage ───────────────────────────────────────────────────────────
        F.col("pipeline_run_id")
         .cast(StringType()),
    )
    # Add Silver processed timestamp
    .withColumn(
        "silver_processed_timestamp",
        get_silver_timestamp(SilverConfig.RUN_TS)
    )
)
 
raw_count = typed_df.count()
logger.info(f"  Rows after type casting: {raw_count:,}")
 
 

In [0]:
typed_df.display()

In [0]:
# =============================================================================
# CELL 5 — DATA QUALITY FILTERS
#
# RULES (from doc Section 6.1 and DataQuality config):
#   - Drop if coin_id is null   (no primary key → can't identify the coin)
#   - Drop if snapshot_date is null (no date → can't partition or merge correctly)
#   - Drop if current_price_usd is null or <= 0 (unusable for any price analysis)
#   - Drop if market_cap_usd is null or <= 0 (unusable for market cap analysis)
#   - Drop if price_change_pct_24h is beyond ±MAX_PRICE_CHANGE_PCT (bad data)
# =============================================================================
 
logger.info("CELL 5: Applying data quality filters")
 
clean_df = (
    typed_df
    .filter(F.col("coin_id").isNotNull())
    .filter(F.col("snapshot_date").isNotNull())
    .filter(
        (F.col("current_price_usd").isNotNull()) &
        (F.col("current_price_usd") > DataQuality.MIN_PRICE_USD)
    )
    .filter(
        (F.col("market_cap_usd").isNotNull()) &
        (F.col("market_cap_usd") > DataQuality.MIN_MARKET_CAP_USD)
    )
    # Sanity check: % change beyond ±10,000% is almost certainly bad data
    .filter(
        F.col("price_change_pct_24h").isNull() |
        (
            (F.col("price_change_pct_24h") >= -DataQuality.MAX_PRICE_CHANGE_PCT) &
            (F.col("price_change_pct_24h") <=  DataQuality.MAX_PRICE_CHANGE_PCT)
        )
    )
)
 
clean_count = clean_df.count()
logger.info(f"  Rows after quality filters: {clean_count:,}")
 
# Raise if too many rows were dropped
validate_drop_rate(
    rows_before  = raw_count,
    rows_after   = clean_count,
    max_fraction = DataQuality.MAX_DROP_FRACTION,
    table_name   = "market_snapshot",
    logger       = logger,
)
 

In [0]:
#  =============================================================================
# CELL 6 — DEDUPLICATE WITHIN THIS BATCH
#
# WHY DEDUPLICATE?
#   The API might rarely return the same coin twice in one response (edge case),
#   or the Bronze table might have two files from the same run (pipeline retry).
#   dropDuplicates on the MERGE key ensures Silver never sees duplicate keys
#   BEFORE the MERGE happens — Delta MERGE would handle it anyway, but this
#   is cleaner and prevents a multi-match warning from Delta.
# =============================================================================
 
logger.info("CELL 6: Deduplicating within batch")
 
dedup_df = clean_df.dropDuplicates(MergeKeys.MARKET_SNAPSHOT)
 
dedup_count = dedup_df.count()
logger.info(f"  Rows after dedup: {dedup_count:,} (dropped {clean_count - dedup_count:,} duplicates)")
 
 

In [0]:
# =============================================================================
# CELL 7 — FINAL COLUMN REORDER
# Reorder to exact Silver schema spec from SilverColumns.MARKET_SNAPSHOT.
# This ensures the Delta table schema is consistent regardless of PySpark
# internal column ordering (which can vary with complex transformations).
# =============================================================================
 
logger.info("CELL 7: Reordering to final Silver schema")
 
final_df = dedup_df.select(*SilverColumns.MARKET_SNAPSHOT)
 
log_schema(final_df, "market_snapshot_final", logger)
 
 
# =============================================================================
# CELL 8 — DELTA MERGE INTO SILVER
# =============================================================================
 
logger.info("CELL 8: MERGE into silver/market_snapshot")
 
merge_stats = delta_merge(
    spark      = spark,
    new_df     = final_df,
    table_path = SilverPaths.MARKET_SNAPSHOT,
    merge_keys = MergeKeys.MARKET_SNAPSHOT,
    logger     = logger,
)
# Update watermark ONLY after successful MERGE.
# If MERGE failed, an exception would have already propagated — we never reach here.
update_watermark(
    spark          = spark,
    source_table   = WatermarkPaths.COINS_MARKETS,
    new_ts         = max_bronze_ts,
    watermark_path = WatermarkPaths.WATERMARK_TABLE,
    run_ts         = SilverConfig.RUN_TS,
    logger         = logger,
)
 

In [0]:
# =============================================================================
# CELL 9 — OPTIMIZE + Z-ORDER
# Z-ORDER BY coin_id + snapshot_date:
#   - Gold queries almost always filter by coin_id (for per-coin aggregations)
#   - snapshot_date is used for daily windowing
#   Together they give maximum data skipping for Gold reads.
# =============================================================================
 
logger.info("CELL 9: OPTIMIZE silver/market_snapshot")
 
spark.sql(f"OPTIMIZE delta.`{SilverPaths.MARKET_SNAPSHOT}` ZORDER BY (coin_id, snapshot_date)")
logger.info("  ✓ OPTIMIZE complete")
 

In [0]:
# =============================================================================
# CELL 10 — RUN LOG + COMPLETION SUMMARY
# =============================================================================
 
summary = {
    "notebook"                 : "02a_silver_market_snapshot",
    "pipeline_run_id"          : SilverConfig.RUN_ID,
    "run_timestamp_utc"        : SilverConfig.RUN_TS.isoformat(),
    "bronze_source"            : BronzePaths.COINS_MARKETS,
    "silver_target"            : SilverPaths.MARKET_SNAPSHOT,
    "rows_from_bronze"         : raw_count,
    "rows_after_quality_filter": clean_count,
    "rows_after_dedup"         : dedup_count,
    "merge_rows_before"        : merge_stats["rows_before"],
    "merge_rows_after"         : merge_stats["rows_after"],
    "merge_rows_inserted"      : merge_stats["rows_inserted"],
    "status"                   : "SUCCESS",
}
 
write_run_log(summary, LogPaths.MARKET_SNAPSHOT, logger)
 
logger.info("=" * 70)
logger.info("Silver: market_snapshot — COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<35}: {v}")
logger.info("=" * 70)
 
dbutils.notebook.exit(json.dumps(summary))